In [ ]:
dataset- https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from google.colab import files
df = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [5]:
df = pd.read_csv(list(df.keys())[0])
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.replace({
    'sentiment': {
        'positive': 1,
        'negative': 0
    }
    }, inplace = True)

/tmp/ipykernel_1243/3940153307.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({


In [7]:
df['review'] = df['review'].str.lower()

In [8]:
df.drop_duplicates(inplace = True)

In [9]:
import re
def remove_html_tags(text):
  pattern = r'<.*?>'
  text = re.sub(pattern, ' ', text)
  return text
df['review'] = df['review'].apply(remove_html_tags)
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming t...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [11]:
punc_pattern = r'[^a-zA-Z\s]'
df['review'] = df['review'].apply(lambda x: re.sub(punc_pattern, ' ', x))

In [12]:
import nltk

In [13]:
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df['review'] = df['review'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [14]:
def remove_urls(text):
  url = r'https?://\S+|www\.\S+'
  return re.sub(url, ' ', text)
df['review'] = df['review'].apply(remove_urls)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [16]:
X = df['review']
y = df['sentiment']

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

In [21]:
tfidf = TfidfVectorizer(max_features = 5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [22]:
logistic_model = LogisticRegression(max_iter=500)
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=500)

In [23]:
y_pred = logistic_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of logistic model: {accuracy: .4f}")

Accuracy of logistic model:  0.8886


In [24]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)

RandomForestClassifier(random_state=42)

In [25]:
y_pred_rf = rf_model.predict(X_test_tfidf)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy of random forest model: {accuracy_rf: .4f}")

Accuracy of random forest model:  0.8364


In [26]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_tfidf, y_train)

DecisionTreeClassifier(random_state=42)

In [27]:
y_pred_dt = dt_model.predict(X_test_tfidf)
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"Accuracy of decision tree model: {accuracy_dt: .4f}")

Accuracy of decision tree model:  0.7071


In [28]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_tfidf, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:36:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [29]:
y_pred_xgb = xgb_model.predict(X_test_tfidf)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy of Xgboost model: {accuracy_xgb: .4f}")

Accuracy of Xgboost model:  0.8460


In [30]:
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_tfidf, y_train)

KNeighborsClassifier()

In [31]:
y_pred_knn = knn_model.predict(X_test_tfidf)
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f"Accuracy of KNN model: {accuracy_knn: .4f}")

Accuracy of KNN model:  0.7395


In [32]:
from sklearn.svm import SVC
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

SVC(kernel='linear', random_state=42)

In [33]:
y_pred_svm = svm_model.predict(X_test_tfidf)
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy of SVM model: {accuracy_svm: .4f}")

Accuracy of SVM model:  0.8873


In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [36]:
train_data, test_data = train_test_split(df, test_size=0.2 ,random_state=42)

In [37]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])
X_train_lstm = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test_lstm = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=200)

In [38]:
Y_train_lstm = train_data['sentiment']
Y_test_lstm = test_data['sentiment']

In [39]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [40]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [41]:
model.fit(X_train_lstm, Y_train_lstm, validation_data=(X_test_lstm, Y_test_lstm), epochs=10, batch_size=64)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 278s 436ms/step - accuracy: 0.8324 - loss: 0.3814 - val_accuracy: 0.8732 - val_loss: 0.3030
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 266s 429ms/step - accuracy: 0.8897 - loss: 0.2749 - val_accuracy: 0.8719 - val_loss: 0.3122
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 269s 435ms/step - accuracy: 0.9057 - loss: 0.2393 - val_accuracy: 0.8744 - val_loss: 0.3226
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 311s 417ms/step - accuracy: 0.9106 - loss: 0.2229 - val_accuracy: 0.8718 - val_loss: 0.3222
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 249s 402ms/step - accuracy: 0.9279 - loss: 0.1850 - val_accuracy: 0.8742 - val_loss: 0.3577
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 285s 438ms/step - accuracy: 0.9361 - loss: 0.1667 - val_accuracy: 0.8479 - val_loss: 0.3921
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 273s 441ms/step - accuracy: 0.9449 - loss: 0.1473 - val_accuracy: 0.8656 - val_loss: 0.3807
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 267s 430ms/step - accuracy: 0.9523 -

In [42]:
pred = model.predict(X_test_lstm)

310/310 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step


In [43]:
y_pred_classes = (pred > 0.5).astype("int32")

In [47]:
label_map = {1: "Positive", 0: "Negative"}
print("Top 10 Predictions")
for i in range(10):
    actual = label_map[y_test.values[i]]
    predicted = label_map[y_pred_classes[i][0]]
    print(f"Sample {i+1}: Actual: {actual} | Predicted: {predicted}")

Top 10 Predictions
Sample 1: Actual: Negative | Predicted: Negative
Sample 2: Actual: Positive | Predicted: Positive
Sample 3: Actual: Negative | Predicted: Positive
Sample 4: Actual: Negative | Predicted: Negative
Sample 5: Actual: Positive | Predicted: Positive
Sample 6: Actual: Positive | Predicted: Positive
Sample 7: Actual: Negative | Predicted: Negative
Sample 8: Actual: Negative | Predicted: Positive
Sample 9: Actual: Negative | Predicted: Negative
Sample 10: Actual: Positive | Predicted: Positive
